# VaR-Filtered Mean Reversion

Backtests a reversal strategy with a historical Value-at-Risk filter.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
VOLATILITY-ADJUSTED MEAN REVERSION STRATEGY (VaR-FILTERED)
===========================================================

STRATEGY OVERVIEW:
-----------------
This strategy exploits mean reversion opportunities ONLY when downside risk is low.
The key insight is that mean reversion works better in stable environments - when
volatility spikes, mean reversion can fail catastrophically.

EXECUTION LOGIC (STRICT T-1 → T):
---------------------------------
Day T-1 (Close):
  - Calculate z-score using data up to T-1
  - Calculate VaR(10%) using historical returns up to T-1
  - Generate signal: Long if (z-score < -1) AND (VaR < 0.02)
  - Store signal (no execution yet)

Day T (Open):
  - Execute pending signal at Open[T]
  - Pay commission ($1 per trade)
  - Update positions

Day T (Close):
  - Value portfolio at Close[T]
  - Calculate returns: Close[T]/Close[T-1] - 1
  - Move to next day

WHY THIS WORKS:
--------------
1. MEAN REVERSION: Assets tend to revert to their mean after extreme deviations
2. VaR FILTER: Only trade when tail risk is low (prevents catastrophic losses)
3. Z-SCORE: Identifies statistical extremes (oversold conditions)
4. RISK CONTROL: VaR acts as volatility regime filter

Author: Trading Strategy Implementation
Date: 2025
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime, timedelta
from typing import Dict, Tuple, Optional, List
from dataclasses import dataclass

warnings.filterwarnings("ignore")


class StrategyConfig:
    """
    Centralized configuration for all strategy parameters.

    CRITICAL: All parameters are configurable here - no hardcoding.
    This allows easy optimization and sensitivity analysis.
    """

    END_DATE = "2024-12-31"
    START_DATE = "2019-12-31"
    INITIAL_CAPITAL = 10000.0
    COMMISSION_PER_TRADE = 1.0
    ZSCORE_WINDOW = 20
    ZSCORE_THRESHOLD = -1.0
    VAR_WINDOW = 20
    VAR_PERCENTILE = 10
    VAR_THRESHOLD = 0.02
    TICKER = "QQQ"
    BENCHMARK = "SPY"
    ROLLING_SHARPE_WINDOW = 126


class DataFetcher:
    """
    Handles all data acquisition from Yahoo Finance.

    CRITICAL: No forward-filling or back-filling of missing data.
    We use ONLY the data that exists to maintain realism.
    """

    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        """
        Fetch historical OHLCV data from Yahoo Finance.

        IMPORTANT:
        - No synthetic data
        - No forward/backward filling
        - Returns raw data only

        Parameters:
        -----------
        ticker : str
            Ticker symbol
        start_date : str
            Start date (YYYY-MM-DD)
        end_date : str
            End date (YYYY-MM-DD)

        Returns:
        --------
        pd.DataFrame with OHLCV data
        """
        buffer_start = (pd.to_datetime(start_date) - timedelta(days=30)).strftime("%Y-%m-%d")
        try:
            response_prices = {"historical": download_price_records(ticker, buffer_start, end_date)}
            data = response_prices
            if "historical" not in data or len(data["historical"]) < 50:
                raise ValueError(f"Insufficient data for {ticker}")
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").reset_index(drop=True)
            df = df.rename(
                columns={
                    "date": "Date",
                    "open": "Open",
                    "high": "High",
                    "low": "Low",
                    "close": "Close",
                    "volume": "Volume",
                }
            )
            df = df[["Date", "Open", "High", "Low", "Close", "Volume"]]
            null_pct = df["Close"].isnull().sum() / len(df)
            if null_pct > 0.05:
                raise ValueError(f"Too much missing data for {ticker}: {null_pct:.1%}")
            print(
                f"✓ Fetched {len(df)} records for {ticker} ({df['Date'].min().date()} to {df['Date'].max().date()})"
            )
            return df
        except Exception as e:
            print(f"✗ Error fetching {ticker}: {str(e)}")
            raise


class FeatureEngine:
    """
    Calculates technical features using STRICT T-1 methodology.

    CRITICAL PRINCIPLE: All calculations for date T use ONLY data up to T-1.
    Features are lagged by 1 day before use in signal generation.
    """

    @staticmethod
    def calculate_log_returns(prices: pd.Series) -> pd.Series:
        """
        Calculate log returns.

        Log returns are time-additive and better for statistical analysis:
        - log(P[t]/P[t-1]) = log(P[t]) - log(P[t-1])
        - More symmetric than simple returns
        - Better for mean reversion analysis

        Parameters:
        -----------
        prices : pd.Series
            Price series

        Returns:
        --------
        pd.Series of log returns
        """
        return np.log(prices / prices.shift(1))

    @staticmethod
    def calculate_zscore(returns: pd.Series, window: int) -> pd.Series:
        """
        Calculate rolling z-score of returns.

        Z-SCORE INTERPRETATION:
        - z < -2: Extremely oversold (2+ std devs below mean)
        - z < -1: Oversold (1+ std dev below mean)
        - -1 < z < 1: Normal range
        - z > 1: Overbought (1+ std dev above mean)
        - z > 2: Extremely overbought

        MEAN REVERSION LOGIC:
        When z-score is very negative, asset is statistically cheap
        and likely to revert upward toward its mean.

        Parameters:
        -----------
        returns : pd.Series
            Return series
        window : int
            Rolling window size

        Returns:
        --------
        pd.Series of z-scores
        """
        rolling_mean = returns.rolling(window=window, min_periods=window).mean()
        rolling_std = returns.rolling(window=window, min_periods=window).std()
        rolling_std = rolling_std.clip(lower=1e-06)
        zscore = (returns - rolling_mean) / rolling_std
        return zscore

    @staticmethod
    def calculate_historical_var(returns: pd.Series, window: int, percentile: float) -> pd.Series:
        """
        Calculate historical Value-at-Risk (VaR).

        VaR INTERPRETATION:
        VaR(10%) = 0.02 means: "In the worst 10% of days, we lose at least 2%"

        WHY VAR AS VOLATILITY FILTER:
        - VaR captures tail risk (worst-case scenarios)
        - Low VaR = stable environment = mean reversion works better
        - High VaR = turbulent market = mean reversion can fail
        - More robust than simple volatility (std) for risk filtering

        CALCULATION METHOD:
        Historical VaR uses empirical distribution (no parametric assumptions)
        - Take rolling window of past returns
        - Calculate specified percentile (e.g., 10th)
        - This is the VaR estimate

        Parameters:
        -----------
        returns : pd.Series
            Return series
        window : int
            Rolling window size
        percentile : float
            Percentile for VaR (e.g., 10 for 10th percentile)

        Returns:
        --------
        pd.Series of VaR values (positive = loss)
        """
        var_series = returns.rolling(window=window, min_periods=window).quantile(percentile / 100.0)
        var_series = -var_series
        return var_series

    @staticmethod
    def prepare_features(df: pd.DataFrame, config: StrategyConfig) -> pd.DataFrame:
        """
        Prepare all features with proper lagging to avoid look-ahead bias.

        EXECUTION FLOW FOR FEATURES:
        1. Calculate raw features from OHLCV data
        2. LAG all features by 1 day (critical!)
        3. At close of day T-1, we know lagged features for T-1
        4. We use T-1 lagged features to generate signal for execution at T

        This ensures that signals for day T use only information
        available at close of day T-1.

        Parameters:
        -----------
        df : pd.DataFrame
            OHLCV data
        config : StrategyConfig
            Strategy configuration

        Returns:
        --------
        pd.DataFrame with lagged features
        """
        data = df.copy()
        data["LogReturns"] = FeatureEngine.calculate_log_returns(data["Close"])
        data["ZScore"] = FeatureEngine.calculate_zscore(data["LogReturns"], config.ZSCORE_WINDOW)
        data["VaR"] = FeatureEngine.calculate_historical_var(
            data["LogReturns"], config.VAR_WINDOW, config.VAR_PERCENTILE
        )
        data["ZScore_Lag"] = data["ZScore"].shift(1)
        data["VaR_Lag"] = data["VaR"].shift(1)
        data["Returns"] = data["Close"].pct_change()
        return data


class SignalGenerator:
    """
    Generates trading signals using T-1 features.

    SIGNAL LOGIC:
    - Long (signal = 1) when:
      1. Z-score < threshold (oversold)
      2. VaR < threshold (low tail risk)
    - Cash (signal = 0) otherwise

    This combines mean reversion opportunity (oversold) with risk control (low VaR).
    """

    def __init__(self, config: StrategyConfig):
        self.config = config

    def generate_signals(self, data: pd.DataFrame) -> pd.DataFrame:
        """
        Generate trading signals using lagged features.

        EXECUTION FLOW:
        - Signal generated at Close[T-1] using lagged features
        - Signal stored in row T-1
        - Signal executed at Open[T]

        Parameters:
        -----------
        data : pd.DataFrame
            Data with lagged features

        Returns:
        --------
        pd.DataFrame with signals
        """
        df = data.copy()
        df["Signal"] = 0
        condition_oversold = df["ZScore_Lag"] < self.config.ZSCORE_THRESHOLD
        condition_low_risk = df["VaR_Lag"] < self.config.VAR_THRESHOLD
        df.loc[condition_oversold & condition_low_risk, "Signal"] = 1
        df["Signal_Change"] = df["Signal"].diff()
        return df


class BacktestEngine:
    """
    Realistic backtesting engine with proper execution logic.

    EXECUTION FLOW:
    Day T-1 (Signal Day):
      - Observe Close[T-1]
      - Calculate features using data up to T-1
      - Generate signal

    Day T (Execution Day):
      - Execute at Open[T]
      - Pay commission
      - Update positions

    Day T (Close):
      - Value portfolio at Close[T]
      - Calculate returns: Close[T]/Close[T-1] - 1

    COMMISSIONS:
    - $1 per trade (entry)
    - $1 per trade (exit)
    - Reduces realized returns
    - Important for realism (eliminates high-frequency artifacts)
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.trades = []

    def run_backtest(self, data: pd.DataFrame) -> Dict:
        """
        Execute backtest with realistic execution.

        Returns:
        --------
        Dict with complete backtest results
        """
        print("\n" + "=" * 80)
        print("EXECUTING BACKTEST (T-1 → T LOGIC)")
        print("=" * 80)
        df = data.copy()
        df["Position"] = 0
        df["Cash"] = self.config.INITIAL_CAPITAL
        df["Holdings"] = 0.0
        df["Portfolio_Value"] = self.config.INITIAL_CAPITAL
        df["Strategy_Returns"] = 0.0
        cash = self.config.INITIAL_CAPITAL
        shares = 0
        position = 0
        for i in range(1, len(df)):
            current_idx = df.index[i]
            prev_idx = df.index[i - 1]
            signal = df.loc[prev_idx, "Signal"]
            open_price = df.loc[current_idx, "Open"]
            close_price = df.loc[current_idx, "Close"]
            if pd.isna(open_price) or pd.isna(close_price):
                df.loc[current_idx, "Position"] = position
                df.loc[current_idx, "Cash"] = cash
                df.loc[current_idx, "Holdings"] = (
                    shares * close_price if not pd.isna(close_price) else 0
                )
                df.loc[current_idx, "Portfolio_Value"] = cash + (
                    shares * close_price if not pd.isna(close_price) else 0
                )
                continue
            if signal == 1 and position == 0:
                available = cash - self.config.COMMISSION_PER_TRADE
                if available > 0:
                    shares = available / open_price
                    cost = shares * open_price + self.config.COMMISSION_PER_TRADE
                    cash -= cost
                    position = 1
                    self.trades.append(
                        {
                            "Date": df.loc[current_idx, "Date"],
                            "Action": "BUY",
                            "Price": open_price,
                            "Shares": shares,
                            "Commission": self.config.COMMISSION_PER_TRADE,
                            "Cash_After": cash,
                        }
                    )
            elif signal == 0 and position == 1:
                proceeds = shares * open_price - self.config.COMMISSION_PER_TRADE
                cash += proceeds
                self.trades.append(
                    {
                        "Date": df.loc[current_idx, "Date"],
                        "Action": "SELL",
                        "Price": open_price,
                        "Shares": shares,
                        "Commission": self.config.COMMISSION_PER_TRADE,
                        "Cash_After": cash,
                    }
                )
                shares = 0
                position = 0
            holdings_value = shares * close_price
            portfolio_value = cash + holdings_value
            if i > 0:
                prev_portfolio_value = df.loc[prev_idx, "Portfolio_Value"]
                if prev_portfolio_value > 0:
                    strategy_return = portfolio_value / prev_portfolio_value - 1
                else:
                    strategy_return = 0
            else:
                strategy_return = 0
            df.loc[current_idx, "Position"] = position
            df.loc[current_idx, "Cash"] = cash
            df.loc[current_idx, "Holdings"] = holdings_value
            df.loc[current_idx, "Portfolio_Value"] = portfolio_value
            df.loc[current_idx, "Strategy_Returns"] = strategy_return
        df["Strategy_Cumulative"] = (1 + df["Strategy_Returns"]).cumprod()
        print(f"\n✓ Backtest completed")
        print(f"  Total trades: {len(self.trades)}")
        print(f"  Final portfolio value: ${df['Portfolio_Value'].iloc[-1]:,.2f}")
        print(
            f"  Total return: {df['Portfolio_Value'].iloc[-1] / self.config.INITIAL_CAPITAL - 1:.2%}"
        )
        return {
            "data": df,
            "trades": self.trades,
            "initial_capital": self.config.INITIAL_CAPITAL,
            "final_value": df["Portfolio_Value"].iloc[-1],
        }

    def add_benchmark(self, results: Dict, benchmark_data: pd.DataFrame) -> Dict:
        """
        Add benchmark buy-and-hold performance.

        BENCHMARK EXECUTION:
        - Buy at first Open with commission
        - Hold until end
        - Sell at last Open with commission

        Parameters:
        -----------
        results : Dict
            Backtest results
        benchmark_data : pd.DataFrame
            Benchmark price data

        Returns:
        --------
        Dict with benchmark added
        """
        df = results["data"]
        bench = benchmark_data[["Date", "Open", "Close"]].rename(
            columns={"Open": "Bench_Open", "Close": "Bench_Close"}
        )
        df = df.merge(bench, on="Date", how="left")
        first_open = df["Bench_Open"].iloc[0]
        if pd.notna(first_open) and first_open > 0:
            available = self.config.INITIAL_CAPITAL - self.config.COMMISSION_PER_TRADE
            shares = available / first_open
            df["Bench_Value"] = shares * df["Bench_Close"]
            last_idx = df.index[-1]
            last_open = df.loc[last_idx, "Bench_Open"]
            if pd.notna(last_open):
                final_value = shares * last_open - self.config.COMMISSION_PER_TRADE
                df.loc[last_idx, "Bench_Value"] = final_value
            df["Bench_Returns"] = df["Bench_Close"].pct_change()
            df["Bench_Cumulative"] = (1 + df["Bench_Returns"]).cumprod()
        else:
            df["Bench_Value"] = self.config.INITIAL_CAPITAL
            df["Bench_Returns"] = 0
            df["Bench_Cumulative"] = 1
        results["data"] = df
        print(f"✓ Benchmark added: Final value ${df['Bench_Value'].iloc[-1]:,.2f}")
        return results


class PerformanceAnalyzer:
    """
    Calculate comprehensive performance metrics.

    Uses realistic annualization (252 trading days) and proper
    risk-adjusted metrics.
    """

    @staticmethod
    def calculate_metrics(results: Dict) -> Dict:
        """
        Calculate all performance metrics.

        METRICS INCLUDE:
        - Total return and CAGR
        - Volatility (annualized)
        - Sharpe ratio (risk-adjusted returns)
        - Sortino ratio (downside risk-adjusted)
        - Calmar ratio (return / max drawdown)
        - Maximum drawdown
        - Hit ratio (% positive trades)
        - Alpha vs benchmark

        Returns:
        --------
        Dict with all metrics
        """
        df = results["data"]
        trades = results["trades"]
        strategy_returns = df["Strategy_Returns"].dropna()
        n_years = len(strategy_returns) / 252
        final_value = df["Portfolio_Value"].iloc[-1]
        initial_value = results["initial_capital"]
        total_return = final_value / initial_value - 1
        cagr = (final_value / initial_value) ** (1 / n_years) - 1 if n_years > 0 else 0
        volatility = strategy_returns.std() * np.sqrt(252)
        sharpe = cagr / volatility if volatility > 0 else 0
        downside_returns = strategy_returns[strategy_returns < 0]
        downside_vol = (
            downside_returns.std() * np.sqrt(252) if len(downside_returns) > 0 else volatility
        )
        sortino = cagr / downside_vol if downside_vol > 0 else 0
        cumulative = (1 + strategy_returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_dd = drawdown.min()
        calmar = cagr / abs(max_dd) if max_dd != 0 else 0
        if len(trades) >= 2:
            buy_trades = [t for t in trades if t["Action"] == "BUY"]
            sell_trades = [t for t in trades if t["Action"] == "SELL"]
            wins = 0
            total_trades = min(len(buy_trades), len(sell_trades))
            for i in range(total_trades):
                buy_price = buy_trades[i]["Price"]
                sell_price = sell_trades[i]["Price"]
                if sell_price > buy_price:
                    wins += 1
            hit_ratio = wins / total_trades if total_trades > 0 else 0
        else:
            hit_ratio = 0
        bench_returns = df["Bench_Returns"].dropna()
        bench_final = df["Bench_Value"].iloc[-1]
        bench_total_return = bench_final / initial_value - 1
        bench_cagr = (bench_final / initial_value) ** (1 / n_years) - 1 if n_years > 0 else 0
        bench_volatility = bench_returns.std() * np.sqrt(252)
        bench_sharpe = bench_cagr / bench_volatility if bench_volatility > 0 else 0
        bench_cumulative = (1 + bench_returns).cumprod()
        bench_running_max = bench_cumulative.expanding().max()
        bench_drawdown = (bench_cumulative - bench_running_max) / bench_running_max
        bench_max_dd = bench_drawdown.min()
        alpha = cagr - bench_cagr
        metrics = {
            "Strategy": {
                "Total Return": total_return,
                "CAGR": cagr,
                "Volatility": volatility,
                "Sharpe Ratio": sharpe,
                "Sortino Ratio": sortino,
                "Calmar Ratio": calmar,
                "Max Drawdown": max_dd,
                "Hit Ratio": hit_ratio,
                "Total Trades": len(trades),
                "Final Value": final_value,
            },
            "Benchmark": {
                "Total Return": bench_total_return,
                "CAGR": bench_cagr,
                "Volatility": bench_volatility,
                "Sharpe Ratio": bench_sharpe,
                "Max Drawdown": bench_max_dd,
                "Final Value": bench_final,
            },
            "Alpha": alpha,
        }
        return metrics

    @staticmethod
    def print_metrics(metrics: Dict):
        """Print formatted metrics table."""
        print("\n" + "=" * 80)
        print("PERFORMANCE METRICS - VOLATILITY-ADJUSTED MEAN REVERSION")
        print("=" * 80)
        print("\nSTRATEGY PERFORMANCE:")
        for name, value in metrics["Strategy"].items():
            if name in ["Total Return", "CAGR", "Volatility", "Max Drawdown"]:
                print(f"  {name:20s}: {value:>10.2%}")
            elif name in ["Sharpe Ratio", "Sortino Ratio", "Calmar Ratio", "Hit Ratio"]:
                print(f"  {name:20s}: {value:>10.2f}")
            elif name == "Total Trades":
                print(f"  {name:20s}: {value:>10d}")
            else:
                print(f"  {name:20s}: ${value:>10,.2f}")
        print("\nBENCHMARK PERFORMANCE:")
        for name, value in metrics["Benchmark"].items():
            if name in ["Total Return", "CAGR", "Volatility", "Max Drawdown"]:
                print(f"  {name:20s}: {value:>10.2%}")
            elif name in ["Sharpe Ratio"]:
                print(f"  {name:20s}: {value:>10.2f}")
            else:
                print(f"  {name:20s}: ${value:>10,.2f}")
        print(f"\nALPHA (vs Benchmark): {metrics['Alpha']:+.2%}")
        print("=" * 80 + "\n")


def create_visualizations(results: Dict, config: StrategyConfig):
    """
    Create comprehensive visualization dashboard.

    PLOTS:
    1. Equity curve (strategy vs benchmark)
    2. Drawdown curve
    3. Rolling 6-month Sharpe ratio
    4. VaR vs actual returns scatter
    """
    df = results["data"]
    fig, axes = plt.subplots(4, 1, figsize=(16, 14))
    ax1 = axes[0]
    ax1.plot(df["Date"], df["Portfolio_Value"], label="Strategy", linewidth=2.5, color="darkblue")
    ax1.plot(
        df["Date"],
        df["Bench_Value"],
        label=f"{config.BENCHMARK} (Benchmark)",
        linewidth=2,
        color="orange",
        alpha=0.7,
    )
    ax1.set_title("Equity Curve: Strategy vs Benchmark", fontsize=14, fontweight="bold")
    ax1.set_ylabel("Portfolio Value ($)", fontsize=12)
    ax1.legend(loc="upper left", fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax2 = axes[1]
    strategy_cumulative = (1 + df["Strategy_Returns"]).cumprod()
    strategy_running_max = strategy_cumulative.expanding().max()
    strategy_dd = (strategy_cumulative - strategy_running_max) / strategy_running_max
    bench_cumulative = (1 + df["Bench_Returns"]).cumprod()
    bench_running_max = bench_cumulative.expanding().max()
    bench_dd = (bench_cumulative - bench_running_max) / bench_running_max
    ax2.fill_between(df["Date"], strategy_dd * 100, 0, alpha=0.7, color="red", label="Strategy DD")
    ax2.fill_between(df["Date"], bench_dd * 100, 0, alpha=0.5, color="orange", label="Benchmark DD")
    ax2.set_title("Drawdown Analysis", fontsize=14, fontweight="bold")
    ax2.set_ylabel("Drawdown (%)", fontsize=12)
    ax2.legend(loc="lower left", fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax3 = axes[2]
    rolling_window = config.ROLLING_SHARPE_WINDOW
    rolling_mean = df["Strategy_Returns"].rolling(rolling_window).mean() * 252
    rolling_std = df["Strategy_Returns"].rolling(rolling_window).std() * np.sqrt(252)
    rolling_sharpe = rolling_mean / rolling_std
    ax3.plot(
        df["Date"],
        rolling_sharpe,
        linewidth=2,
        color="green",
        label=f"{rolling_window}-day Rolling Sharpe",
    )
    ax3.axhline(y=0, color="black", linestyle="--", linewidth=1, alpha=0.5)
    ax3.axhline(y=1, color="gray", linestyle=":", linewidth=1, alpha=0.5)
    ax3.set_title("Rolling Sharpe Ratio (6-month)", fontsize=14, fontweight="bold")
    ax3.set_ylabel("Sharpe Ratio", fontsize=12)
    ax3.legend(loc="upper left", fontsize=11)
    ax3.grid(True, alpha=0.3)
    ax4 = axes[3]
    valid_data = df[["VaR_Lag", "LogReturns"]].dropna()
    ax4.scatter(
        valid_data["VaR_Lag"] * 100,
        -valid_data["LogReturns"] * 100,
        alpha=0.5,
        s=20,
        color="purple",
    )
    max_var = valid_data["VaR_Lag"].max() * 100
    ax4.plot([0, max_var], [0, max_var], "r--", linewidth=2, alpha=0.7, label="VaR = Actual Loss")
    ax4.axvline(
        x=config.VAR_THRESHOLD * 100,
        color="green",
        linestyle=":",
        linewidth=2,
        label=f"VaR Threshold ({config.VAR_THRESHOLD:.1%})",
    )
    ax4.set_title("VaR Filter: Predicted vs Actual Losses", fontsize=14, fontweight="bold")
    ax4.set_xlabel("VaR (10th percentile) - %", fontsize=12)
    ax4.set_ylabel("Actual Loss - %", fontsize=12)
    ax4.legend(loc="upper left", fontsize=11)
    ax4.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def main():
    """Main execution function."""
    print("=" * 80)
    print("VOLATILITY-ADJUSTED MEAN REVERSION STRATEGY (VaR-FILTERED)")
    print("=" * 80)
    config = StrategyConfig()
    print(f"\nCONFIGURATION:")
    print(f"  Period: {config.START_DATE} to {config.END_DATE}")
    print(f"  Ticker: {config.TICKER}")
    print(f"  Benchmark: {config.BENCHMARK}")
    print(f"  Initial Capital: ${config.INITIAL_CAPITAL:,.0f}")
    print(f"  Commission: ${config.COMMISSION_PER_TRADE} per trade")
    print(f"\n  Z-Score Window: {config.ZSCORE_WINDOW} days")
    print(f"  Z-Score Threshold: {config.ZSCORE_THRESHOLD}")
    print(f"  VaR Window: {config.VAR_WINDOW} days")
    print(f"  VaR Percentile: {config.VAR_PERCENTILE}%")
    print(f"  VaR Threshold: {config.VAR_THRESHOLD:.1%}")
    print("\n" + "-" * 80)
    print("FETCHING DATA")
    print("-" * 80)
    fetcher = DataFetcher()
    ticker_data = fetcher.fetch_historical_data(config.TICKER, config.START_DATE, config.END_DATE)
    benchmark_data = fetcher.fetch_historical_data(
        config.BENCHMARK, config.START_DATE, config.END_DATE
    )
    print("\n" + "-" * 80)
    print("CALCULATING FEATURES (T-1 METHODOLOGY)")
    print("-" * 80)
    data_with_features = FeatureEngine.prepare_features(ticker_data, config)
    print(f"✓ Features calculated: {len(data_with_features)} observations")
    print("\n" + "-" * 80)
    print("GENERATING SIGNALS")
    print("-" * 80)
    signal_gen = SignalGenerator(config)
    data_with_signals = signal_gen.generate_signals(data_with_features)
    num_long_signals = (data_with_signals["Signal"] == 1).sum()
    print(f"✓ Long signals: {num_long_signals}")
    print("\n" + "-" * 80)
    print("RUNNING BACKTEST")
    print("-" * 80)
    engine = BacktestEngine(config)
    results = engine.run_backtest(data_with_signals)
    results = engine.add_benchmark(results, benchmark_data)
    print("\n" + "-" * 80)
    print("CALCULATING PERFORMANCE METRICS")
    print("-" * 80)
    metrics = PerformanceAnalyzer.calculate_metrics(results)
    PerformanceAnalyzer.print_metrics(metrics)
    print("\n" + "-" * 80)
    print("GENERATING VISUALIZATIONS")
    print("-" * 80)
    create_visualizations(results, config)
    print("\n" + "=" * 80)
    print("EXECUTION COMPLETE")
    print("=" * 80)
    return (results, metrics)


if __name__ == "__main__":
    results, metrics = main()
